# Лекција 18: Осигурање AI агената помоћу криптографских рачуна

## Практични блок

Овај блок води кроз четири вежбе:

1. **Потпишите свој први рачун** за позив алата агента и верификујте га.
2. **Манипулишите рачуном** и пратите како верификација не успева.
3. **Изградите ланац од три рачуна** и потврдите интегритет ланца.
4. **Обмотајте позив алата из Microsoft Agent Framework-а** тако да свака радња издаје рачун.

Све криптографске примитиве су увезене из добро одржаваних библијотека (`pynacl` за Ed25519, `jcs` за RFC 8785 канонски JSON, `hashlib` из стандардне Python библиотеке за SHA-256). Логика рачуна је чист Python који можете читати и мењати.

Покрените ћелије по реду. Свако поглавље је кратко и самостално.


## Подешавање

Инсталирајте две зависности. Обе имају дозволе које су либералне (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Помоћни алати

Ова два помоћна алата рукују кодирањем base64url (без допуне) и хеширањем SHA-256 произвољних објеката. Они омогућавају да остатак нотебука остане фокусиран на саму логику рецепта.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Одељак 1: Потпишите свој први рачун

Замислите да је наш агент за **Contoso Travel** управо прегледао летове од Сиднеја до Лос Анђелеса за купца. Желимо да снимимо изјаву гатеваја о овом позиву алата како би будући ревизор могао да је провери под посебно закрепљеним кључем издаваоца.

### Корак 1.1: Генеришите кључ за потписивање

У производњи, потписни кључ агента би се налазио у уређају за безбедност хардвера (HSM), Azure Key Vault-у или сличном заштићеном складишту. За ову лекцију генеришемо нови кључ у меморији.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Корак 1.2: Изградите садржај потврде

Садржај садржи све што желимо да потврда потврди: ко је деловао, који алат, са којим аргументима, шта је резултат, под којом политиком и када. Хеширамо аргументе и резултат уместо да их укључујемо у линију како потврда не би процурила осетљив садржај.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Четврт 1.3: Потпишите и саставите рачун

Три корака:

1. Канонизујте payload користећи JCS како би две имплементације које производе исти логички рачун произвеле бајт-идентичне бајтове.
2. Потпишите канонске бајтове директно са Ed25519 приватним кључем. PureEdDSA унутрашње хешира поруку, тако да би додатни прехеш променио протокол.

Потпис се затим прилаже оригиналном payload-у да би се произвео коначан рачун.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Корак 1.4: Потврдите потврду

Потврда је обрнути процес. Уклонимо потпис, поново израчунамо канонске бајтове и проверавамо потпис у односу на јавни кључ издаваоца.

Потврда је офлајн: не позивамо сервис и нема приступа мрежи. Ипак, потребна је још једна ствар осим потврде: јавни кључ издаваоца, добијен путем канала којем ревизор већ верује, као што је JWK скуп који издаваоц објављује или вредност унапред причвршћена. Поље `public_key` унутар потврде не може играти ту улогу, јер ко год напише потврду, такође пише и то поље. Одељак 2 показује фалсификат који ово спречава.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Требало би да видите `Receipt is valid: True`, `Pre-hashed receipt valid: False` и `Malformed receipts rejected: True`. Позитивни случај доказује да директан-JCS пут потписивања ради; негативна контрола чини правило опсега потписа извршним уместо да га остави као прозу; последња линија показује неисправан улаз који враћа `False` уместо да баци изузетак.


## Одељак 2: Мешање са рачуном

Цела поента рачуна је да су они отпорни на неовлашћене измене. Хајде да то докажемо.

Модификоваћемо тачно један знак на рачуну и посматрати како проверa не успева.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Шта се управо десило?

Када смо променили `policy_id`, канонски бајтови су се променили. Потпис (који је био преко оригиналних канонских бајтова) више није усклађен. Верификација исправно враћа `False`.

Нема начина да се промени потписано поље, што значи било шта изван `signature` објекта, а да се у исто време потписница пријема верификује помоћу јавног кључа издаваоца, осим ако нападач нема приватни кључ издаваоца. Сам `signature` објекат није потписан: `sig` је потпис, а `verify_receipt` игнорише `public_key`, па једино уређивање тог поља не мења ништа. Док год је приватни кључ у витрину кључева и верификатори користе јавно објављен кључ издаваоца, промена у потписаном пољу је немогуће прикрити. Тај услов је важан: следећа ћелија показује модификовану потписницу која се савршено верификује помоћу другог кључа.

Испробајте сами: измените `tool_name` или `agent_id` или `timestamp` у ћелији изнад и покрените поново. Свака промена у потписаном пољу производи неважећу потписницу.


### Манипулација коју потпис не може да ухвати

Нападач који је приказан горе није поново потписао. Онај који је пажљивији би: изменио признаницу,
потписао је кључем који је сам генерисао, и ставио тај кључ у
`public_key` поље. Резултат је признаница чији је потпис потпуно аутентичан
за њен садржај.

Покрените ово и погледајте који од два провере то уочава.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Требало би да видите `True` затим `False`. Три `assert` реда чине да то буде
провера: ћелија избацује грешку ако лажна копија икада пролази верификацију против
кључа издаваоца.

Лажна копија је унутрашње конзистентна. Њен потпис заиста одговара њеном
садржају и кључу који носи, тако да верификатор који прочита кључ из
рачуна пријављује га као ваљан. Није ништа на рачуну што изгледа погрешно, јер
ништа на њему није неконзистентно.

Једина ствар која је открива је провера против кључа који нападач није могао
да утиче. Зато `verify_receipt` узима кључ као аргумент: функција са потписом
`verify_receipt(receipt)` не може бити безбедно написана,
јер је једини кључ до кога може доћи онај који је нападач доставио.

Исти рачун се чува као `sample_receipts/04_resigned_receipt.json` ако желите
да га прегледате ван бележнице.


## Одељак 3: Повежите рачуне у ланац

Један рачун штити једну акцију. Већина агената предузима много акција. Да бисмо целу секвенцу учинили отпорном на фалсификовање, вежемо сваки рачун за претходни укључујући хаш претходног рачуна у терет новог рачуна.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ако неко уклони или преуреди рачун унутар представљене секвенце, следећа карика пуца. Откривање скраћивања или преписивања наставка који контролише потписник такође захтева закључану текућу главу, спољни проверни пункт или аутентификовани популациони ангажман.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Сада прекини низ манипулацијом средњег рачуна и поново изврши проверу. Израњени рачун не пролази проверу потписа, И следећи рачун не пролази проверу ланчаног линка (јер његов `previous_receipt_hash` више не одговара хешу модификованог средњег рачуна).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Рачун 0 још увек верификује (није измењен и нема претходник на који би се ослањао). Рачун 1 не пролази проверу дигиталног печата јер смо изменили `tool_args_hash`. Рачун 2 не пролази проверу ланца веза јер је његов `previous_receipt_hash` израчунат на основу оригиналног (сада изменјеног) рачуна 1.

Чак и ако нападач поново потпише измењени рачун 1, нови потпис неће проћи проверу са кључем издаваоца осим ако не поседује приватни кључ издаваоца (поновно потписивање сопственим кључем не успева, као што је показанo у Одељку 2), а непоклапање у ланцу веза у рачуну 2 и даље би открило манипулацију. Да би прикрио измену, нападач би морао поново потписати сваки рачун од тачке измене па надаље, што захтева поседовање приватног кључа издаваоца.


## Одељак 4: Обмотавање позива алата агента са потписивањем рачуна

У реалном постављању, не желите да сваки аутор агента памти да позове `make_receipt`. Желите да потписивање рачуна буде аутоматско за сваки позив алата.

Ево најједноставнијег узорка: класа омотача која узима било коју позиву функцију алата и враћа верзију која емитује рачун. Ово се адаптира на било који агентски оквир, укључујући Microsoft Agent Framework (`agent_framework.foundry`).

Ако немате подешен Microsoft Foundry пројекат, локални мок испод и даље демонстрира овај узорак.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Интеграција са Microsoft Agent Framework-ом

Обавијач `ReceiptedTool` изнад је оквирно неутралан. Да бисте га користили у агенту изграђеном са Microsoft Agent Framework-ом, региструјте умотану функцију као алат. Овде је скица (заменили бисте мок стварном регистрацијом алата Microsoft Foundry):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Агентски оквир не мора да зна ништа о рачунима. Потписивање рачуна је умотано око алата, а није уграђено у оквир. Ово је начин да додате порекло постојећем агентском коду без преписивања агента.


## Резиме и изазов за продужетак

Ви имате:

- Генерисан Ed25519 клучни пар.
- Креиран и потписан рачун за позив агенцијског алата.
- Верификован рачун ван мреже користећи само јавни кључ издаваоца, добијен независно од рачуна.
- Измењен рачун и уочен неуспех верификације.
- Креирана ланчана секвенца од три рачуна.
- Измењен средњи део ланца и уочени неуспеси како у потпису тако и у ланчаном везивању.
- Обмотана функција алата са аутоматским потписивањем рачуна.

**Изазов за продужетак.** Прошirite схему рачуна са `request_id` пољем (UUID за дистрибуирано праћење). Ажурирајте `make_receipt` да га укључује и потврдите да рачуни и даље пролазе комплетну верификацију. Затим модификујте поље након потписивања и уверите се да верификација не успева. Ово вас приморава да унутар себе схватите како сваки бајт канонског енкодинга доприноси потпису.

**Важна граница.** Под фиксним кључем и контекстом ланца, рачуни доказују атрибуцију потписа, интегритет садржаја и везу са приказаним претходником. Они НЕ доказују извршење, извор истине, поуздано време, потпуну команду, евалуацију политике или исправност. Рачуни су темељ. Управљање је систем који градите на врху.

Прочитајте упутство поново имајући на уму ту границу. Најчешћа грешка тимова у вези са рачунима је претпоставка да „имамо рачуне“ значи „ми смо управљани“. То није тако. Рачуни чине понашање агента аудиторним. Они га не чине исправним.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
